### Import Libraries
Import the required libraries and define the functions as provided:

In [1]:
import os
import tensorflow as tf

# Suppress TensorFlow logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

2024-06-14 14:05:54.922947: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-14 14:05:54.923002: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-14 14:05:54.924698: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-06-14 14:05:54.932645: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import datetime as dt
import logging
import traceback
import json
import psycopg2
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# TensorFlow and Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from imblearn.over_sampling import SMOTE

# Logging setup
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
console_handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

# Database connection function
def connection_db(path_to_json):
    with open(path_to_json) as config_file:
        db_params = json.load(config_file)
    try:
        conn = psycopg2.connect(**db_params)
        cursor = conn.cursor()
        logger.info('Connection to database established')
        return conn, cursor
    except Exception as e:
        logger.error('Connection to database could not be established.', exc_info=True)

# Data fetching function
def fetch_data(conn, cursor):
    get_data = '''SELECT * FROM group14_warehouse.regression_data'''
    cursor.execute(get_data)
    logger.info('Data pulled from warehouse')
    df = pd.DataFrame(cursor.fetchall(), columns=['datetime_s', 'datetime_e', 'event_cat', 'event_sev', 'speed', 'end_speed',
                                                  'maxwaarde', 'streetname', 'rain_intensity', 'temperature', 'windspeed',
                                                  'speed_limit', 'light_condition', 'accident_sev', 'accident_prob'])
    conn.close()
    logger.info('Connection to database closed')
    return df

# Data type fixing function
def fix_dtypes(df, cats, dats):
    for column in dats:
        df[column] = pd.to_datetime(df[column])
    for column in cats:
        encoder = LabelEncoder()
        df[column] = encoder.fit_transform(df[column])
    return df

# Remove duplicates function
def remove_duplicates(df):
    duplicates = df.duplicated(keep=False)
    df = df[~duplicates]
    return df

# Create X and y function
def create_xy(df):
    X = df.drop(['accident_sev', 'accident_prob'], axis=1)
    y = df['accident_prob']
    return X, y

# Split data into train, validation, and test sets
def split_tvt(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=0)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=0)
    return X_train, X_val, X_test, y_train, y_val, y_test

# Normalize data function
def normalize_data(X_train, X_val=None, X_test=None, scaler=None):
    if scaler is None:
        scaler = StandardScaler()
        scaler.fit(X_train)
    X_train_norm = scaler.transform(X_train)
    X_val_norm = scaler.transform(X_val) if X_val is not None else None
    X_test_norm = scaler.transform(X_test) if X_test is not None else None
    return X_train_norm, X_val_norm, X_test_norm, scaler

# Define functions with .loc to avoid SettingWithCopyWarning
def set_speed_limit(df):
    df.loc[:, 'speed_limit'] = df['speed_limit'].astype(str).str.replace("Un", "0", regex=False).str.replace("No", "0", regex=False)
    df = df[pd.to_numeric(df['speed_limit'], errors='coerce').notnull()]
    df.loc[:, 'speed_limit'] = df['speed_limit'].astype(int)
    return df

def create_additional_datetime_features(df):
    holidays_nl = [
        # Add holidays as per your data
    ]
    df.loc[:, 'is_holiday'] = df["datetime_s"].dt.date.isin([pd.to_datetime(holiday[0]).date() for holiday in holidays_nl])
    df.loc[:, 'weekday'] = df['datetime_s'].dt.dayofweek
    df.loc[:, 'hour'] = df['datetime_s'].dt.hour
    df.loc[:, 'duration'] = df['datetime_e'] - df['datetime_s']
    df.loc[:, 'duration_in_s'] = df['duration'].dt.total_seconds().astype(int)
    return df

def calculate_gforce(df):
    df.loc[df['event_cat'] == 3, 'maxspeed'] = df['maxwaarde']
    df.loc[df['event_cat'] != 3, 'gforce'] = df['maxwaarde']
    df.loc[:, 'delta_v'] = df['maxspeed'] - df['speed']
    df.loc[:, 'acceleration'] = df['delta_v'] / df['duration_in_s']
    df.loc[:, 'calculated_gforce'] = df['acceleration'] / 9.81
    df.loc[:, 'gforce'] = df['gforce'].fillna(df['calculated_gforce'])
    df.drop(columns=['delta_v', 'acceleration', 'calculated_gforce'], inplace=True)
    return df

def feature_engineering(df):
    df = set_speed_limit(df)
    df = create_additional_datetime_features(df)
    df = calculate_gforce(df)
    df.drop(columns=['datetime_s', 'datetime_e', 'maxwaarde', 'duration', 'maxspeed'], inplace=True)
    return df

def encode_categoricals(df, categorical_columns):
    for column in categorical_columns:
        encoder = LabelEncoder()
        df[column] = encoder.fit_transform(df[column])
    return df

# Plot loss and accuracy
def loss_plotter(H):
    plt.plot(range(1, len(H.history['loss']) + 1), H.history['loss'], label='train_loss')
    plt.plot(range(1, len(H.history['val_loss']) + 1), H.history['val_loss'], label='val_loss')
    plt.title('Model Loss')
    plt.ylabel('loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

def accuracy_plotter(H):
    plt.plot(range(1, len(H.history['accuracy']) + 1), H.history['accuracy'], label='train_acc')
    plt.plot(range(1, len(H.history['val_accuracy']) + 1), H.history['val_accuracy'], label='val_acc')
    plt.title('Model Accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

### Load and Preprocess Data
Load and preprocess the data using the provided functions:

In [4]:
json_path = '../db_config.json'

# Load and preprocess data
conn, cursor = connection_db(json_path)
df = fetch_data(conn, cursor)
df = remove_duplicates(df)
df = fix_dtypes(df, ['event_cat', 'event_sev', 'streetname', 'light_condition', 'accident_sev'], ['datetime_s', 'datetime_e'])
df = feature_engineering(df)
df = encode_categoricals(df, ['event_cat', 'event_sev', 'streetname', 'light_condition'])
X, y = create_xy(df)

# Split data
X_train, X_val, X_test, y_train, y_val, y_test = split_tvt(X, y)
X_train, X_val, X_test, scaler = normalize_data(X_train, X_val, X_test)

# Convert to tensors
X_train = tf.convert_to_tensor(X_train, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)
X_val = tf.convert_to_tensor(X_val, dtype=tf.float32)
y_val = tf.convert_to_tensor(y_val, dtype=tf.float32)
X_test = tf.convert_to_tensor(X_test, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test, dtype=tf.float32)

# Inspect the dataset
print(df.head())
print(df.describe())
print(df.info())
print(df.isnull().sum())
print(y.value_counts())

2024-06-14 14:11:57,351 - __main__ - INFO - Connection to database established
INFO: Connection to database established
2024-06-14 14:12:06,979 - __main__ - INFO - Data pulled from warehouse
INFO: Data pulled from warehouse
2024-06-14 14:12:40,298 - __main__ - INFO - Connection to database closed
INFO: Connection to database closed
/tmp/ipykernel_1319670/1092353480.py:99: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, 'is_holiday'] = df["datetime_s"].dt.date.isin([pd.to_datetime(holiday[0]).date() for holiday in holidays_nl])
/tmp/ipykernel_1319670/1092353480.py:100: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the d

    event_cat  event_sev      speed  end_speed  streetname  rain_intensity  \
7           2          2  30.577536   37.01491         105             0.0   
8           2          2  30.577536   37.01491         105             0.0   
9           2          2  30.577536   37.01491         105             0.0   
18          2          2  30.577536   37.01491         105             0.0   
19          2          2  30.577536   37.01491         105             0.0   

    temperature  windspeed speed_limit  light_condition  accident_sev  \
7           8.8      12.87          70                1             2   
8           8.8      12.87          50                2             2   
9           8.8      12.87           0                0             2   
18          8.8      12.87          30                1             1   
19          8.8      12.87           0                1             1   

    accident_prob  is_holiday  weekday  hour  duration_in_s    gforce  
7        3.467174   

### Define and Train the Deep Learning Model
Define and train a new deep learning model:

In [ ]:
def build_advanced_model(input_dim):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.4))
    model.add(Dense(512, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.4))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

input_dim = X_train.shape[1]
model = build_advanced_model(input_dim)

# Define callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True)

# Train the model
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, batch_size=32, callbacks=[early_stopping, model_checkpoint])

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/tensorflow/python/util/dispatch.py:1260: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(None, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return dispatch_target(*args, **kwargs)
2024-06-14 14:13:04.207880: I external/local_xla/xla/service/service.cc:168] XLA service 0x7efc60b05790 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-06-14 14:13:04.207916: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2024-06-14 14:13:04.214206: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-06-14 14:13:04.253614: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8906
I0000 00:00:

32455/32455 [==============================] - 188s 6ms/step - loss: nan - accuracy: 0.0000e+00 - val_loss: nan - val_accuracy: 0.0000e+00
Epoch 2/100
25619/32455 [======================>.......] - ETA: 38s - loss: nan - accuracy: 0.0000e+00